In [0]:
use catalog identifier(:use_catalog)

In [0]:
-- Get the Start Date for the Tagging Process
-- We want to default to the last date in the existing data set and NOT re-process any calls that already exist in the target table. 
-- This date is INCLUSIVE only if a call on this day does not exist in the target table. 
declare or replace variable start_date date;
set variable start_date = coalesce(try_cast(:sae_start_date as date), (select date(max(`time`)) from [schema_name].copilot_competitive_info));

In [0]:
-- Get the End Date for the CoPilot AI Tagging Process
-- We want to default to today
-- This date is EXCLUSIVE - it will NOT be included in the data set
declare or replace variable end_date date;
set variable end_date = coalesce(try_cast(:sae_end_date as date), current_date);

In [0]:
-- Note the start date and end date
select
  session.start_date,
  session.end_date
;

In [0]:
CREATE OR REPLACE TABLE [schema_name].relationship_risk_identifier_stg
TBLPROPERTIES ('delta.columnMapping.mode' = 'name') AS
SELECT
    `id`
  , `time`
  , to_json(transcript_list) as `transcript`,
  ai_query(
    '[llm-model-name]',
    CONCAT(
      '## Role & Objective

You are a churn-risk detection analyst for [Company]\'s Customer Success team. You read a single customer call transcript and identify whether any of **three specific renewal risks** are present, based strictly on what is said in the transcript.

You detect ONLY these three risks. Do not flag anything else.

1. **Economic Buyer/ Decision Maker Churn**
2.**Champion Churn**
3. **Technical Influencer Churn** 


### Core rules

- **Ground all judgments strictly in transcript text.** Flag a risk only when explicit statements in the transcript support it. Never infer risk from silence, tone, pauses, unstated assumptions, or external context.
- **Require a verbatim quote for every flag.** Extract an exact, word-for-word quote from the transcript to justify every flagged risk. If you cannot extract a direct quote, do not flag the risk. No quote = no flag.
- **Validate speaker authority before flagging. Evaluate statements spoken directly by the customer, or direct customer quotes restated verbatim by the account team. Ignore internal rep speculation, sales hypotheses, or rep-generated worry.
- **Evaluate each risk independently.** Scan for all qualifying risks without artificial caps. A single transcript may yield zero, one, or multiple flags based on what the quotes support.
- **Default to an empty response when no risks exist.** Return an empty result when a call contains no qualifying evidence. Never force, guess, or approximate a match—clean calls are expected.
- **Map quotes precisely to persona definitions. Categorize risks using these explicit functional boundaries. Assign both risks if a departing individual explicitly holds multiple documented roles (e.g., both Champion and Technical Lead).

### Risk definitions and detection signals

**1. Economic Buyer/ Decision Maker Churn**
The economic buyer or decision maker who authorized the original [Company] purchase has **changed**. The successor carries no institutional memory of why [Company] was selected, no emotional investment in the relationship, and no obligation to honor prior commitments. They may arrive with a mandate to rationalize the vendor stack, demonstrate independent judgment, or justify their appointment through competitive review. The transcript must contain explicit evidence of this transition.
Detection signals (each requires evidence of a CHANGE in who holds authority):
- The executive who signed the original purchase has left, been promoted out, or moved to a different role — and a replacement has taken over.
- A new CIO, CISO, or VP Security has been explicitly described as "new" or "recently appointed" — they have no prior [Company] relationship.
- The new DM is conducting a "clean slate" technology review or re-evaluating all vendors.
- A previously uninvolved executive (e.g., new CFO, new CTO) has inserted themselves into the renewal decision — and the transcript makes clear this involvement is new.
- The new DM has a stated prior relationship or preference for a competitor.
*Do NOT flag:* A customer simply mentioning who holds budget authority (e.g., "my CEO approves all spend," "everything flows through my manager") is normal organizational hierarchy — NOT churn. Flag only when the transcript provides evidence that the person in that role has changed or is newly asserting absolute control that did not exist before.
*Disambiguation:* This is about **spend authority** — the person who signs or vetoes the PO. Advocacy without budget power = Champion. Hands-on usage without budget power = Technical Influencer.

**2. Champion Churn**
The internal advocate who uses [Company], understands its value, and communicates that value upward has **departed or disengaged**. A replacement contact has no [Company] history, no context, and often no existing relationship with the CSM. The transcript must contain explicit mention of a personnel change or disengagement.
Detection signals (flag when the transcript mentions any of these):
- Primary [Company] contact has changed roles, departments, or companies
- CSMs last email to them was unreturned for 14+ days
- Champion not attending scheduled calls or declining meeting invites
- Champion has stopped contributing to Clari call transcripts or account activity
- A new contact is introduced by the customer as the "new person handling [Company]"
*Do NOT flag:* A champion simply being absent from one call, a customer asking a routine question, or a customer being unfamiliar with a single feature. These do not indicate the advocate has departed — only explicit statements about departure, role change, or prolonged disengagement qualify.
*Disambiguation:* This is about **internal advocacy** — the person who sold [Company]\'s value upward. Budget authority without advocacy = Economic Buyer. Platform operation without political influence = Technical Influencer.

**3. Technical Influencer Churn**
The practitioner who knows how to run [Company], configure scans, interpret VPR, manage integrations, and build dashboards has **departed**. The replacement might be technically able but has no [Company] experience. The transcript must contain explicit mention of a personnel change or loss of platform knowledge.
Detection signals (flag when the transcript mentions any of these):
- The security analyst or engineer who runs the day-to-day [Company] operations has left
- New technical contact has no [Company] training or certification
- Customer asks for "retraining" or "starting over" on [Company]
*Do NOT flag:* A customer asking a how-to question about a specific feature, requesting help with a new module they haven\'t used before, or a routine training request unrelated to personnel change. Unfamiliarity with a single feature is not Technical Influencer Churn — the signal requires evidence that the person who previously operated the platform is gone.
*Disambiguation:* This is about **daily platform operation** — the person who runs scans and builds dashboards. Budget authority without hands-on use = Economic Buyer. Political advocacy without platform use = Champion. If one person holds multiple roles, flag both.

### Output format

Return a JSON object with the following keys:

- **`Is_CSM_Renewal_Risk_Present`** (boolean) — `true` if at least one of the three risks is present in the transcript; `false` if none qualify.
- **`CSM_Renewal_Risk_Summary`** (string) — a concise summary of the context in which the risk(s) were captured: what was happening in the conversation that surfaced them. If no risks are present, return an empty string.
- **`CSM_Renewal_Risk_Customer_Sentiment`** — the customer\'s overall sentiment toward [Company] across the whole call. One of `Positive`, `Negative`, `Neutral`:
  - **Positive** — the customer expresses satisfaction, praise, or advocacy for [Company].
  - **Negative** — the customer expresses frustration, dissatisfaction, or criticism of [Company].
  - **Neutral** — mixed, transactional, or no clear sentiment either way.
  - Judge sentiment from what the *customer* says, not the [Company] rep. Sentiment is independent of risk: a customer can be positive about [Company] and still present a renewal risk (e.g., new appointment), and vice versa.
- **`CSM_Renewal_Detected_Risks`** (array) — one entry per flagged risk; an empty array when none qualify. Each entry contains:
  - **`CSM_Renewal_Risk_Category`** — exactly one of: `Economic Buyer, Decision Maker churn`, `Champion Churn`, `Technical Influencer Churn`.
  - **`CSM_Renewal_Risk_Reasoning`** — 1-3 sentences. State the observation and which detection signal it maps to. Factual; do not overstate certainty.
  - **`CSM_Renewal_Risk_Key_Verbatim_Quote`** — verbatim from the transcript. Do not paraphrase. You may stitch together non-contiguous fragments if multiple supporting quotes exist.
  - **`CSM_Renewal_Risk_Strength`** — a judgment of how severe the captured risk is: how much threat the detected risk(s) pose to the renewal. One of `N/A`, `Low`, `Medium`, `High`:
  - **Low** — minimal threat to the renewal: a personnel change is mentioned but the departing individual\'s role is peripheral, or a warm handoff is already underway with a knowledgeable replacement.
  - **Medium** — a real but not decisive threat: a key relationship holder (economic buyer, champion, or technical influencer) has departed or is departing, but the replacement is engaged, willing to learn, or continuity measures are in place — no decision to leave or replace [Company] has been stated.
  - **High** — a serious threat: a key relationship holder has left with no warm handoff, the replacement has no [Company] context and shows no engagement, multiple relationship roles are affected simultaneously, or the new decision maker is actively evaluating competitors or conducting a clean-slate review.
  - Rate conservatively and ground it in what was actually said. When no risks are detected, the level is `N/A`.
  - **`CSM_Renewal_Risk_Confidence`** - A judgement of how confident you are in your assessment of this risk mention. One of "Low","Medium","High".


### Example

Transcript excerpt:
> **Customer:** Yeah so Mark actually left the company about three weeks ago. I\'m the new person handling [Company] — I\'m still getting up to speed on everything.
> **Customer:** He was the one running all the scans and dashboards. I\'ve never used [Company] before so we\'ll probably need some retraining sessions.

Output:
```json
{
  "Is_CSM_Renewal_Risk_Present": true,
  "CSM_Renewal_Risk_Summary": "During a routine check-in the customer revealed their primary security contact has left the company and a new person with no [Company] experience has taken over.",
  "CSM_Renewal_Detected_Risks": [
    {
      "CSM_Renewal_Risk_Category": "Champion Churn",
      "CSM_Renewal_Risk_Reasoning": "The customer\'s primary [Company] contact has left the organization and a new contact has been introduced as the replacement, matching the \'new person handling [Company]\' detection signal.",
      "CSM_Renewal_Risk_Key_Verbatim_Quote": "Yeah so Mark actually left the company about three weeks ago. I\'m the new person handling [Company] — I\'m still getting up to speed on everything.",
      "CSM_Renewal_Risk_Strength": "Medium",
      "CSM_Renewal_Risk_Confidence": "High"
    },
    {
      "CSM_Renewal_Risk_Category": "Technical Influencer Churn",
      "CSM_Renewal_Risk_Reasoning": "The departed individual ran day-to-day [Company] operations (scans and dashboards). The replacement explicitly states no [Company] experience and requests retraining, matching the \'customer asks for retraining or starting over\' detection signal.",
      "CSM_Renewal_Risk_Key_Verbatim_Quote": "He was the one running all the scans and dashboards. I\'ve never used [Company] before so we\'ll probably need some retraining sessions.",
      "CSM_Renewal_Risk_Strength": "High",
      "CSM_Renewal_Risk_Confidence": "High"
    }
  ],
  "CSM_Renewal_Risk_Customer_Sentiment": "Neutral"
}
```

## Transcript
',
      to_json(`transcript_list`)
    ),
    responseFormat => '{
  "type": "json_schema",
  "json_schema": {
    "name": "csm_at_risk_renewal_extraction",
    "schema": {
      "type": "object",
      "properties": {
  "Is_CSM_Renewal_Risk_Present": {"type": "boolean"},
  "CSM_Renewal_Risk_Summary": {"type": "string"},
  "CSM_Renewal_Detected_Risks": {
    "type": "array",
    "items": {
      "type": "object",
      "properties": {
        "CSM_Renewal_Risk_Category": {"type": "string", "enum": ["Economic Buyer, Decision Maker Churn","Champion Churn","Technical Influencer Churn"]},
        "CSM_Renewal_Risk_Reasoning": {"type": "string"},
        "CSM_Renewal_Risk_Key_Verbatim_Quote": {"type": "string"},
        "CSM_Renewal_Risk_Strength": {"type": "string", "enum": ["Low","Medium","High"]},
        "CSM_Renewal_Risk_Confidence": {"type": "string", "enum": ["Low","Medium","High"]}
      },
      "required": ["CSM_Renewal_Risk_Category", "CSM_Renewal_Risk_Reasoning", "CSM_Renewal_Risk_Key_Verbatim_Quote", "CSM_Renewal_Risk_Strength", "CSM_Renewal_Risk_Confidence"],
      "additionalProperties": false
    }
  },
  "CSM_Renewal_Risk_Customer_Sentiment": {"type": "string", "enum": ["Positive","Negative","Neutral"]}
},
"required": ["Is_CSM_Renewal_Risk_Present", "CSM_Renewal_Risk_Summary", "CSM_Renewal_Detected_Risks", "CSM_Renewal_Risk_Customer_Sentiment"],


      "additionalProperties": false
    },
    "strict": true
  }
}',
    modelParameters => named_struct('max_tokens', 8192)
  ) AS response
FROM `[catalog_name]`.`[schema_name]`.`call_details`
WHERE `transcript_text` IS NOT NULL
and array_size(filter(joinedParticipants, x -> x.email not ilike '%[company].com%' OR x.email is null)) > 0
and date(`time`) >= '2026-07-28'
and id not in (select id from [schema_name].relationship_risk_identifier)



In [0]:
MERGE INTO [schema_name].relationship_risk_identifier AS target
USING (
  SELECT *
  FROM [schema_name].relationship_risk_identifier_stg
) AS source
  ON target.id = source.id
WHEN NOT MATCHED BY TARGET THEN INSERT *
;

In [0]:
-- Explode the detected risks array into one row per risk for easy review
CREATE OR REPLACE TABLE [schema_name].relationship_risk_identifier_exploded AS
SELECT
    t.`id`
  , t.`time`
  , t.response:Is_CSM_Renewal_Risk_Present::boolean AS is_risk_present
  , t.response:CSM_Renewal_Risk_Summary::string AS risk_summary
  , t.response:CSM_Renewal_Risk_Customer_Sentiment::string AS customer_sentiment
  , r:CSM_Renewal_Risk_Category::string AS risk_category
  , r:CSM_Renewal_Risk_Strength::string AS risk_strength
  , r:CSM_Renewal_Risk_Confidence::string AS risk_confidence
  , r:CSM_Renewal_Risk_Reasoning::string AS risk_reasoning
  , r:CSM_Renewal_Risk_Key_Verbatim_Quote::string AS risk_verbatim_quote
  , t.transcript
FROM [schema_name].relationship_risk_identifier t
LATERAL VIEW explode(parse_json(t.response):CSM_Renewal_Detected_Risks::array<variant>) e AS r